
Credit for these materials go to:
> *DS Python for GIS and Geoscience*  
> *September, 2024*
>
> *© 2024, Joris Van den Bossche and Stijn Van Hoey  (<mailto:jorisvandenbossche@gmail.com>, <mailto:stijnvanhoey@gmail.com>). Licensed under [CC BY 4.0 Creative Commons](http://creativecommons.org/licenses/by/4.0/)*


In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("CP101_LAB07.ipynb")

In [ ]:
!pip install contextily
!pip install "geopandas[all]"

In [ ]:
# Data for this lab ships alongside the notebook (data/raw/ folder below).
import os
assert os.path.isdir('data/raw'), "Expected the data/raw/ folder next to this notebook."
print('Data folder found.')

In [ ]:
%pwd

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# CP101: Lab 7 - GeoPandas II & III: Spatial Predicates, Joins, and Operations

This week we will advance our spatial skills in GeoPandas. We start by joining ACS data to
census tract geometry, working in a real coordinate system, and building a per-capita
measure from it. Then we practice different types of spatial joins, operations, and
overlays on a new dataset. Let's jump right in!

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Joining ACS data to census tracts, and a first choropleth

Lab 6 loaded a TIGER shapefile as a GeoDataFrame and made one plain plot: every tract in
California, in one colour. This section attaches real attribute data to
it, reprojects it into a coordinate system built for measuring instead of storing, and
turns a raw population count into a per-capita number that is fair to compare across
tracts of very different size.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

tracts = pd.read_csv(
    'data/raw/alameda_tracts23.csv', dtype={'state': str, 'county': str, 'tract': str}
)
tracts['geoid'] = tracts['state'] + tracts['county'] + tracts['tract']
tracts['median_household_income'] = tracts['median_household_income'].replace(
    -666666666, np.nan
)

california_tracts = gpd.read_file('data/raw/tl_2023_06_tract.zip')
alameda_geometry = california_tracts[california_tracts['COUNTYFP'] == '001']

print('Attribute rows (from Lab 6):', len(tracts))
print('Tract shapes in Alameda:    ', len(alameda_geometry))

### ✏️ Exercise 1: attach the numbers to the shapes

Two steps, the same ones Lab 6 set up for you.

1. You already have `alameda_geometry`, the Alameda rows of `california_tracts`.
2. Merge `tracts` onto it, into `alameda_geo`.

TIGER calls its key `GEOID` and your column is called `geoid`, so pass `left_on=` and
`right_on=` rather than `on=`. Put the **GeoDataFrame on the left**, so the result keeps
its geometry and stays a GeoDataFrame rather than collapsing to a plain table.

In [ ]:
alameda_geo = ...

print('Tract shapes for Alameda: ', len(alameda_geometry))
print('Attribute rows to attach: ', len(tracts))
print('Rows after the join:      ', len(alameda_geo))
print('Rows lost:                ', len(tracts) - len(alameda_geo))
print()
print('Result type:', type(alameda_geo).__name__)

In [ ]:
grader.check("q_geometry_join")

### Check what the merge did to your column names

Both frames carried a column called `NAME`, and they held different things: TIGER's is the
bare tract number, `4227`, and the ACS table's is the full "Census Tract 4227.03; Alameda
County; California". `.merge()` cannot keep two columns with the same name, so it kept
both and renamed them `NAME_x` (left, TIGER) and `NAME_y` (right, the attribute table).

In [ ]:
print("Is there still a column called 'NAME'?", 'NAME' in alameda_geo.columns)
print("'NAME_x' (from TIGER):", 'NAME_x' in alameda_geo.columns)
print("'NAME_y' (from the ACS table):", 'NAME_y' in alameda_geo.columns)

### Measuring in a real coordinate system

`alameda_geo.crs` is still `EPSG:4269`, the same NAD83 geographic CRS the shapefile shipped
with. Geographic CRSs store coordinates as degrees of longitude and latitude, which is
exactly what you want for storing and joining — GEOIDs do not care what units the shapes
are in — and exactly what you do **not** want for measuring. A degree of longitude covers a
different real distance at every latitude, so an area or a distance computed directly from
`EPSG:4269` coordinates is not a real area or distance at all.

`EPSG:3310`, California (Teale) Albers, is a **projected** CRS built for this state: an
equal-area projection whose units are metres. `.to_crs()` reprojects a GeoDataFrame's
geometry from one CRS to another without touching any of its other columns.

In [ ]:
alameda_projected = alameda_geo.to_crs(epsg=3310)

alameda_geo['area_sq_km'] = alameda_projected.geometry.area / 1e6

print(alameda_geo[['GEOID', 'area_sq_km']].sort_values('area_sq_km').head())
print()
print(alameda_geo['area_sq_km'].describe())

### A per-capita measure: population density

`population` alone is not a fair way to compare tracts. The Census Bureau draws tract
boundaries to hold roughly similar **numbers of people**, not similar amounts of land, so a
379-tract county has some tracts the size of a few blocks and others the size of a small
town. Dividing by area turns the raw count into a **density**, a per-capita measure that is
comparable across tracts of any size.

### ✏️ Exercise 2: the densest tract in the county

Build `alameda_geo['pop_density']`: `population` divided by `area_sq_km`. Then find
`densest_geoid`, the `GEOID` of the tract with the highest density.

In [ ]:
alameda_geo['pop_density'] = ...

densest_geoid = ...
print('Densest tract:              ', densest_geoid)
print('People per square kilometre:', round(alameda_geo['pop_density'].max(), 1))

In [ ]:
grader.check("q_density")

### A first choropleth

`.plot()` takes a `column=` argument and colours each shape by a value instead of drawing
every row in one colour. The left panel below is what Lab 6 left you with; the right one
carries the density you just built.

In [ ]:
figure, (left, right) = plt.subplots(1, 2, figsize=(12, 6))

alameda_geo.plot(ax=left)
left.set_title('One colour (Lab 6)')
left.set_axis_off()

alameda_geo.plot(ax=right, column='pop_density', cmap='YlOrRd', legend=True)
right.set_title('Population density (people / km²)')
right.set_axis_off()

plt.tight_layout()
plt.show()

The right-hand map already says something: people are not spread evenly across Alameda
County's tracts. One tract in particular does not look like the others: no colour at all,
not even the lowest shade.

### ✏️ Exercise 3: the tract that is not a polygon

One of the 379 Alameda tracts is a `MultiPolygon`. Find it. Build `odd_one` by masking
`alameda_geo` on `.geom_type`, and keep `GEOID`, `NAME_x`, `population` and
`median_household_income`. Before you run it, guess what kind of tract has several separate
pieces and no single boundary.

In [ ]:
odd_one = alameda_geo.loc[..., ['GEOID', 'NAME_x', 'population', 'median_household_income']]

print(odd_one.to_string(index=False))

In [ ]:
grader.check("q_multipolygon")

Census Tract 9900 is the Bureau's code for a **water tract**: all of Alameda County's share
of the Bay, drawn as three separate pieces of open water with no land at all, which is why
it needs a `MultiPolygon`. There is nobody there to count, which is also why its density is
`0` rather than missing — `population` is a real zero here, not a sentinel.

That is the habit this section is building: a row in a GeoDataFrame is a place, and when a
number about it looks strange, the shape often explains the number.

From here, the rest of this lab moves from *attributes attached to shapes* to
*relationships between shapes themselves* — spatial predicates, spatial joins, and
overlays — on a new dataset: countries, cities, and rivers.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Spatial relationships and joins

In [ ]:
import os
%matplotlib inline
import pandas as pd
import geopandas
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily
from shapely.geometry import LineString

In [ ]:
# Read the shapefiles
countries = gpd.read_file("data/raw/ne_110m_admin_0_countries")
cities = gpd.read_file("data/raw/ne_110m_populated_places")
rivers = gpd.read_file("data/raw/ne_50m_rivers_lake_centerlines")

The files that we are using have come as shapefiles. In keeping with our conventions, let's first convert these to parquet files and read them as parquet files.

Save each of your gdfs above using the format below:

```
gdf.to_parquet("path/to/your/file.parquet")
```
Then, read your parquet files with the following format:

```
gdf = geopandas.read_parquet("path/to/your/file.parquet")
```



In [ ]:
# Convert to parquet
countries.to_parquet('ne_110m_admin_0_countries')
cities.to_parquet('ne_110m_populated_places')
rivers.to_parquet('ne_50m_rivers_lake_centerlines')

# Read your parquet files
countries = gpd.read_parquet('ne_110m_admin_0_countries')
cities = gpd.read_parquet('ne_110m_populated_places')
rivers = gpd.read_parquet('ne_50m_rivers_lake_centerlines')


<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Spatial relationships

An important aspect of geospatial data is that we can look at *spatial relationships*: how two spatial objects relate to each other (whether they overlap, intersect, contain, .. one another).

The topological, set-theoretic relationships in GIS are typically based on the DE-9IM model. See https://en.wikipedia.org/wiki/Spatial_relation for more information.

![alt text](https://upload.wikimedia.org/wikipedia/commons/5/55/TopologicSpatialRelarions2.png)

![](../img/TopologicSpatialRelarions2.png)
(Image by [Krauss, CC BY-SA 3.0](https://en.wikipedia.org/wiki/Spatial_relation#/media/File:TopologicSpatialRelarions2.png))

### Relationships between individual objects

Let's first create some small toy spatial objects:

A polygon <small>(note: we use `.item()` here to to extract the scalar geometry object from the GeoSeries of length 1)</small>:

In [ ]:
belgium = countries.loc[countries['name'] == 'Belgium', 'geometry'].item()

Two points:

In [ ]:
paris = cities.loc[cities['name'] == 'Paris', 'geometry'].item()
brussels = cities.loc[cities['name'] == 'Brussels', 'geometry'].item()

And a linestring:

In [ ]:
from shapely.geometry import LineString
line = LineString([paris, brussels])

Let's visualize those 4 geometry objects together (I only put them in a GeoSeries to easily display them together with the geopandas `.plot()` method):

In [ ]:
gpd.GeoSeries([belgium, paris, brussels, line]).plot(cmap='tab10')

You can recognize the abstract shape of Belgium.

Brussels, the capital of Belgium, is thus located within Belgium. This is a spatial relationship, and we can test this using the individual shapely geometry objects as follow:

In [ ]:
brussels.within(belgium)

And using the reverse, Belgium contains Brussels:

In [ ]:
belgium.contains(brussels)

On the other hand, Paris is not located in Belgium:

In [ ]:
belgium.contains(paris)

In [ ]:
paris.within(belgium)

The straight line we draw from Paris to Brussels is not fully located within Belgium, but it does intersect with it:

In [ ]:
belgium.contains(line)

In [ ]:
line.intersects(belgium)

### Spatial relationships with GeoDataFrames

The same methods that are available on individual `shapely` geometries as we have seen above, are also available as methods on `GeoSeries` / `GeoDataFrame` objects.

For example, if we call the `contains` method on the world dataset with the `paris` point, it will do this spatial check for each country in the `world` dataframe:

In [ ]:
countries.contains(paris)

Because the above gives us a boolean result, we can use that to filter the dataframe:

In [ ]:
countries[countries.contains(paris)]

And indeed, France is the only country in the world in which Paris is located.

Another example, extracting the linestring of the Amazon river in South America, we can query through which countries the river flows:

In [ ]:
amazon = rivers[rivers['name'] == 'Amazonas'].geometry.item()

In [ ]:
countries[countries.crosses(amazon)]  # or .intersects

<div class="alert alert-info" style="font-size:120%">

**REFERENCE**:

Overview of the different functions to check spatial relationships (*spatial predicate functions*):

* `equals`
* `contains`
* `crosses`
* `disjoint`
* `intersects`
* `overlaps`
* `touches`
* `within`
* `covers`
* `covered_by`


See https://shapely.readthedocs.io/en/stable/manual.html#predicates-and-relationships for an overview of those methods.

See https://en.wikipedia.org/wiki/DE-9IM for all details on the semantics of those operations.

</div>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Let's practice!

We will again use the Paris datasets to exercise. Let's start importing them again, and directly converting both to the local projected CRS:

In [ ]:
districts = gpd.read_file("data/raw/paris_districts.geojson").to_crs(epsg=2154)
stations = gpd.read_file("data/raw/paris_bike_stations.geojson").to_crs(epsg=2154)

<div class="alert alert-success">

**EXERCISE 4: The Eiffel Tower**

The Eiffel Tower is an iron lattice tower built in the 19th century, and is probably the most iconic view of Paris.

The location of the Eiffel Tower is: x of 648237.3 and y of 6862271.9

* Create a Shapely point object with the coordinates of the Eiffel Tower and assign it to a variable called `eiffel_tower`. Print the result.
* Check if the Eiffel Tower is located within the Montparnasse district (provided).
* Check if the Montparnasse district contains the bike station location.
* Calculate the distance between the Eiffel Tower and the bike station (note: in this case, the distance is returned in meters).


<details><summary>Hints</summary>

* The `Point` class is available in the `shapely.geometry` submodule
* Creating a point can be done by passing the x and y coordinates to the `Point()` constructor.
* The `within()` method checks if the object is located within the passed geometry (used as `geometry1.within(geometry2)`).
* The `contains()` method checks if the object contains the passed geometry (used as `geometry1.contains(geometry2)`).
* To calculate the distance between two geometries, the `distance()` method of one of the geometries can be used.

</details>

</div>

In [ ]:
# Import the Point geometry
from shapely.geometry import Point

In [ ]:
districts

In [ ]:
# Accessing the Montparnasse geometry (Polygon)
district_montparnasse = districts.loc[districts['district_name'] == 'Montparnasse', 'geometry'].item()
bike_station = stations.loc[stations['name'] == '14033 - DAGUERRE GASSENDI', 'geometry'].item()

In [ ]:
eiffel_tower = ...
print(eiffel_tower)

# Check if the Eiffel Tower is located within the Montparnasse district (provided)
...

# Check if the Montparnasse district contains the bike station location (provided)
...

# Calculate the distance between the Eiffel Tower and the bike station
...

In [ ]:
grader.check("q_eiffel_point")

<div class="alert alert-success">

**EXERCISE 5: In which district in the Eiffel Tower located?**

In previous exercise, we constructed a `Point` geometry for its location, and we checked that it was not located in the Montparnasse district. Let's now determine in which of the districts of Paris it *is* located.

* Create a boolean mask (or filter) indicating whether each district contains the Eiffel Tower or not. Call the result `mask`.
* Filter the `districts` dataframe with the boolean mask and print the result.


<details><summary>Hints</summary>

* To check for each polygon in the districts dataset if it contains a single point, we can use the `contains()` method of the `districts` GeoDataFrame.
* Filtering the rows of a DataFrame based on a condition can be done by passing the boolean mask into `df[..]`.

</details>

</div>

In [ ]:
mask = ...
districts[mask]

In [ ]:
grader.check("q_eiffel_district")

<div class="alert alert-success">

**EXERCISE 6: How far is the closest bike station?**

Now, we might be interested in the bike stations nearby the Eiffel Tower. To explore them, let's visualize the Eiffel Tower itself as well as the bikes stations within 1km.

To do this, we can calculate the distance to the Eiffel Tower for each of the stations. Based on this result, we can then create a mask that takes `True` if the station is within 1km, and `False` otherwise, and use it to filter the stations GeoDataFrame. Finally, we make a visualization of this subset.

* Calculate the distance to the Eiffel Tower for each station, and call the result `dist_eiffel`.
* Print the distance to the closest station (which is the minimum of `dist_eiffel`).
* Select the rows the `stations` GeoDataFrame where the distance to the Eiffel Tower is less than 1 km (note that the distance is in meters). Call the result `stations_eiffel`.

<details><summary>Hints</summary>

* The `.distance()` method of a GeoDataFrame works element-wise: it calculates the distance between each geometry in the GeoDataFrame and the geometry passed to the method.
* A Series has a `.min()` method to calculate the minimum value.
* To create a boolean mask based on a condition, we can do e.g. `s < 100`.

</details>

</div>

In [ ]:
dist_eiffel = ...
print(dist_eiffel.min())
stations_eiffel = ...

In [ ]:
grader.check("q_eiffel_distance")

In [ ]:
# (OPTIONAL) Run the code below to make a plot of the close-by bike stations using matplotlib and contextily, to do so, just make sure you have named your variables the same as in the instructions above.
fig, ax = plt.subplots(figsize=(8, 8))
stations_eiffel.plot(ax=ax)
gpd.GeoSeries([eiffel_tower], crs='EPSG:2154').plot(ax=ax, color='red')
contextily.add_basemap(ax, crs=stations_eiffel.crs)
ax.set_axis_off()

In [ ]:
# Run the code below to make a plot of the close-by bike stations using matplotlib and contextily
m = stations_eiffel.explore(marker_kwds=dict(radius=5))
gpd.GeoSeries([eiffel_tower], crs='EPSG:2154').explore(m=m, color='red', marker_kwds=dict(radius=5))

---

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Spatial joins

In the previous section of this notebook, we could use the spatial relationship methods to check in which country a certain city was located. But what if we wanted to perform this same operation for every city and country? For example, we might want to know for each city in which country it is located.  

In tabular jargon, this would imply adding a column to our cities dataframe with the name of the country in which it is located. Since country name is contained in the countries dataset, we need to combine - or "join" - information from both datasets. Joining on location (rather than on a shared column) is called a "spatial join".

So here we will do:

- Based on the `countries` and `cities` dataframes, determine for each city the country in which it is located.
- To solve this problem, we will use the the concept of a "spatial join" operation: combining information of geospatial datasets based on their spatial relationship.

### Recap - joining dataframes

Pandas provides functionality to join or merge dataframes in different ways, see https://chrisalbon.com/python/data_wrangling/pandas_join_merge_dataframe/ for an overview and https://pandas.pydata.org/pandas-docs/stable/merging.html for the full documentation.

To illustrate the concept of joining the information of two dataframes with pandas, let's take a small subset of our `cities` and `countries` datasets:

In [ ]:
cities2 = cities[cities['name'].isin(['Bern', 'Brussels', 'London', 'Paris'])].copy()
cities2['iso_a3'] = ['CHE', 'BEL', 'GBR', 'FRA']

In [ ]:
cities2

In [ ]:
countries2 = countries[['iso_a3', 'name', 'continent']]
countries2.head()

We added a 'iso_a3' column to the `cities` dataset, indicating a code of the country of the city. This country code is also present in the `countries` dataset, which allows us to merge those two dataframes based on the common column.

Joining the `cities` dataframe with `countries` will transfer extra information about the countries (the full name, the continent) to the `cities` dataframe, based on a common key:

In [ ]:
cities2.merge(countries2, on='iso_a3')

**But** for this illustrative example we added the common column manually, it is not present in the original dataset. However, we can still know how to join those two datasets based on their spatial coordinates.

### Recap - spatial relationships between objects

In the previous section, we have seen the notion of spatial relationships between geometry objects: within, contains, intersects, ...

In this case, we know that each of the cities is located *within* one of the countries, or the other way around that each country can *contain* multiple cities.

We can test such relationships using the methods we have seen in the previous notebook:

In [ ]:
france = countries.loc[countries['name'] == 'France', 'geometry'].item()

In [ ]:
cities.within(france)

The above gives us a boolean series, indicating for each point in our `cities` dataframe whether it is located within the area of France or not.  
Because this is a boolean series as result, we can use it to filter the original dataframe to only show those cities that are actually within France:

In [ ]:
cities[cities.within(france)]

We could now repeat the above analysis for each of the countries, and add a column to the `cities` dataframe indicating this country. However, that would be tedious to do manually, and is also exactly what the spatial join operation provides us.

*(note: the above result is incorrect, but this is just because of the coarse-ness of the countries dataset)*

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Spatial join operation

<div class="alert alert-info" style="font-size:120%">

**SPATIAL JOIN** = *transferring attributes from one layer to another based on their spatial relationship* <br>


Different parts of this operations:

* The GeoDataFrame to which we want add information
* The GeoDataFrame that contains the information we want to add
* The spatial relationship ("predicate") we want to use to match both datasets ('intersects', 'contains', 'within')
* The type of join: left or inner join


![](../img/illustration-spatial-join.svg)

</div>

In this case, we want to join the `cities` dataframe with the information of the `countries` dataframe, based on the spatial relationship between both datasets.

We use the [`geopandas.sjoin`](http://geopandas.readthedocs.io/en/latest/reference/geopandas.sjoin.html) function:

In [ ]:
joined = gpd.sjoin(cities, countries, predicate='within', how='left')

In [ ]:
joined

In [ ]:
joined[joined["name_right"] == "France"]

In [ ]:
joined['continent'].value_counts()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Let's practice!

We will again use the Paris datasets to do some exercises. Let's start importing them:

In [ ]:
districts = gpd.read_file("data/raw/paris_districts.geojson").to_crs(epsg=2154)
stations = gpd.read_file("data/raw/paris_bike_stations.geojson").to_crs(epsg=2154)

<div class="alert alert-success">

**EXERCISE 7: Which district is each bike station in?**

* Determine for each bike station in which district it is located (using a spatial join!). Call the result `joined`.

<details><summary>Hints</summary>

- The `geopandas.sjoin()` function takes as first argument the dataframe to which we want to add information, and as second argument the dataframe that contains this additional information.

</details>

</div>

In [ ]:
joined = ...

In [ ]:
grader.check("q_district_join")

In [ ]:
joined

<div class="alert alert-success">

**EXERCISE 8: Map of tree density by district (I)**

Using a dataset of all trees in public spaces in Paris, the goal is to make a map of the tree density by district. We first need to find out how many trees each district contains, which we will do in this exercise. In the following exercise, we will use this result to calculate the density and create a map.

To obtain the tree count by district, we first need to know in which district each tree is located, which we can do with a spatial join. Then, using the result of the spatial join, we will calculate the number of trees located in each district using the pandas 'group-by' functionality.

- Import the trees dataset `"paris_trees.gpkg"` and call the result `trees`. Also read the districts dataset we have seen previously (`"paris_districts.geojson"`), and call this `districts`. Convert the districts dataset to the same CRS as the trees dataset.
- Add a column with the `'district_name'` to the trees dataset using a spatial join. Call the result `joined`.

<details><summary>Hints</summary>

- Remember, we can perform a spatial join with the `geopandas.sjoin()` function.
- `geopandas.sjoin()` takes as first argument the dataframe to which we want to add information, and as second argument the dataframe that contains this additional information.
- The `op` argument is used to specify which spatial relationship between both dataframes we want to use for joining (options are `'intersects'`, `'contains'`, `'within'`).

</details>

</div>

Load `trees`, spatially join it into `districts` (matching Exercise 7's pattern), storing the result back into `joined`. Then, for Exercise 9:

<div class="alert alert-success">

**EXERCISE 9: Map of tree density by district (II)**

Calculate the number of trees located in each district: group the `joined` DataFrame by the `'district_name'` column, and calculate the size of each group. Call the resulting Series `trees_by_district`.  <br>

We then convert `trees_by_district` to a DataFrame for the next exercise.

<details><summary>Hints</summary>

- The general group-by syntax in pandas is: `df.groupby('key').aggregation_method()`, substituting 'key' and 'aggregation_method' with the appropriate column name and method.
- To know the size of groups, we can use the `.size()` method.

</details>

</div>

In [ ]:
trees = ...
joined = ...
trees_by_district = ...

In [ ]:
grader.check("q_trees_by_district")

In [ ]:
# Convert the series to a DataFrame and specify column name
trees_by_district = trees_by_district.to_frame(name='n_trees')

In [ ]:
# Inspect the result
trees_by_district.head()

<!-- BEGIN QUESTION -->

<div class="alert alert-success">

**EXERCISE 10: Map of tree density by district (III)**

Now we have obtained the number of trees by district, we can make the map of the districts colored by the tree density.

For this, we first need to merge the number of trees in each district we calculated in the previous step (`trees_by_district`) back to the districts dataset. We will use the [`pd.merge()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.merge.html) function to join two dataframes based on a common column.

Since not all districts have the same size, we should compare the tree density for the visualisation: the number of trees relative to the area.

- Use the `pd.merge()` function to merge `districts` and `trees_by_district` dataframes on the `'district_name'` column. Call the result `districts_trees`.
- Add a column `'n_trees_per_area'` to the `districts_trees` dataframe, based on the `'n_trees'` column divided by the area.
- Make a plot of the `districts_trees` dataframe, using the `'n_trees_per_area'` column to determine the color of the polygons.


<details><summary>Hints</summary>

- The pandas `pd.merge()` function takes the two dataframes you want to merge as the first two arguments.
- The column name on which you want to merge both datasets can be specified with the `on` keyword.
- Accessing a column of a DataFrame can be done with `df['col']`, while adding a column to a DataFrame can be done with `df['new_col'] = values` where `values` can be the result of a computation.
- Remember, the area of each geometry in a GeoSeries or GeoDataFrame can be retrieved using the `area` attribute. So considering a GeoDataFrame `gdf`, then `gdf.geometry.area` will return a Series with the area of each geometry.
- We can use the `.plot()` method of a GeoDataFrame to make a visualization of the geometries.
- For using one of the columns of the GeoDataFrame to determine the fill color, use the `column=` keyword.


</details>

</div>

<!-- END QUESTION -->

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Spatial operations and overlays: creating new geometries

In the previous exercises we have seen how to identify and use the spatial relationships between geometries. In the next exercises, we will see how to create new geometries based on those relationships.

In [ ]:
# defining the same example geometries as in the previous notebook
belgium = countries.loc[countries['name'] == 'Belgium', 'geometry'].item()
brussels = cities.loc[cities['name'] == 'Brussels', 'geometry'].item()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Spatial operations

Next to the spatial predicates that return boolean values, Shapely and GeoPandas also provide operations that return new geometric objects.

**Binary operations:**

<table><tr>
<td> <img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQoH421s379K207Ij7OHdDGzZ2bJgtVCqtR_Q&s"/> </td>
<td> <img src="https://shapely.readthedocs.io/en/stable/_images/intersection-sym-difference.png"/> </td>
</tr>
<tr>
<td> <img src="https://shapely.readthedocs.io/en/stable/_images/union.png"/> </td>
<td> <img src="https://shapely.readthedocs.io/en/stable/_images/difference.png"/> </td>
</tr></table>

**Buffer:**

<table><tr>
<td> <img src="https://geopandas.org/en/latest/_images/buffer.png"/> </td>



See https://shapely.readthedocs.io/en/stable/manual.html#spatial-analysis-methods for more details.

For example, using the toy data from above, let's construct a buffer around Brussels (which returns a Polygon):

In [ ]:
gpd.GeoSeries([belgium, brussels.buffer(1)]).plot(alpha=0.5, cmap='tab10')

and now take the intersection, union or difference of those two polygons:

In [ ]:
brussels.buffer(1).intersection(belgium)

In [ ]:
brussels.buffer(1).union(belgium)

In [ ]:
brussels.buffer(1).difference(belgium)

### Spatial operations with GeoPandas

Above we showed how to create a new geometry based on two individual shapely geometries. The same operations can be extended to GeoPandas. Given a GeoDataFrame, we can calculate the intersection, union or difference of each of the geometries with another geometry.

Let's look at an example with a subset of the countries. We have a GeoDataFrame with the country polygons of Africa, and  now consider a rectangular polygon, representing an area around the equator:

In [ ]:
africa = countries[countries.continent == 'Africa']

In [ ]:
box = LineString([(-10, 0), (50, 0)]).buffer(10, cap_style=3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
africa.plot(ax=ax, facecolor='none', edgecolor='k')
gpd.GeoSeries([box]).plot(ax=ax, facecolor='C0', edgecolor='k', alpha=0.5)

The intersection method of the GeoDataFrame will now calculate the intersection with the rectangle for each of the geometries of the africa GeoDataFrame element-wise. Note that for many of the countries, those that do not overlap with the rectangle, this will be an empty geometry:

In [ ]:
africa_intersection = africa.intersection(box)
africa_intersection.head()

What is returned is a new GeoSeries of the same length as the original dataframe, containing one row per country, but now containing only the intersection. In this example, the last element shown is an empty polygon, as that country was not overlapping with the box.

In [ ]:
# plot the intersection
africa_intersection.plot()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Unary union and dissolve

Another useful method is `union_all()` (in older GeoPandas versions this is available as the `unary_union` attribute), which converts the set of geometry objects in a GeoDataFrame into a single geometry object by taking the union of all those geometries.

For example, we can construct a single Shapely geometry object for the Africa continent:

In [ ]:
africa_countries = countries[countries['continent'] == 'Africa']

In [ ]:
africa = africa_countries.union_all()

In [ ]:
africa

In [ ]:
print(str(africa)[:100])

Alternatively, you might want to take the unary union of a set of geometries but *grouped* by one of the attributes of the GeoDataFrame (so basically doing "groupby" + "unary_union"). For this operation, GeoPandas provides the `dissolve()` method:

In [ ]:
continents = countries.dissolve(by="continent")   # , aggfunc="sum"

In [ ]:
continents

<div class="alert alert-info" style="font-size:120%">

**REMEMBER**:

GeoPandas (and Shapely for the individual objects) provide a whole lot of basic methods to analyze the geospatial data (distance, length, centroid, boundary, convex_hull, simplify, transform, ....), much more than what we can touch in this tutorial.

An overview of all methods provided by GeoPandas can be found here: https://geopandas.org/en/latest/docs/reference.html


</div>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Let's practice!

<div class="alert alert-success">

**EXERCISE 11: What are the districts close to the Seine?**

Below, the coordinates for the Seine river in the neighborhood of Paris are provided as a GeoJSON-like feature dictionary (created at http://geojson.io).

Based on this `seine` object, we want to know which districts are located close (maximum 150 m) to the Seine.

* Create a buffer of 150 m around the Seine.
* Check which districts intersect with this buffered object.
* Make a visualization of the districts indicating which districts are located close to the Seine.

</div>

In [ ]:
districts = gpd.read_file("data/raw/paris_districts.geojson").to_crs(epsg=2154)

In [ ]:
# created a line with http://geojson.io
s_seine = gpd.GeoDataFrame.from_features({"type":"FeatureCollection","features":[{"type":"Feature","properties":{},"geometry":{"type":"LineString","coordinates":[[2.408924102783203,48.805619828930226],[2.4092674255371094,48.81703747481909],[2.3927879333496094,48.82325391133874],[2.360687255859375,48.84912860497674],[2.338714599609375,48.85827758964043],[2.318115234375,48.8641501307046],[2.298717498779297,48.863246707697],[2.2913360595703125,48.859519915404825],[2.2594070434570312,48.8311646245967],[2.2436141967773438,48.82325391133874],[2.236919403076172,48.82347994904826],[2.227306365966797,48.828339513221444],[2.2224998474121094,48.83862215329593],[2.2254180908203125,48.84856379804802],[2.2240447998046875,48.85409863123821],[2.230224609375,48.867989496547864],[2.260265350341797,48.89192242750887],[2.300262451171875,48.910203080780285]]}}]},
                                               crs='EPSG:4326')

In [ ]:
# convert to local UTM zone
s_seine_utm = s_seine.to_crs(epsg=2154)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(20, 10))
districts.plot(ax=ax, color='grey', alpha=0.4, edgecolor='k')
s_seine_utm.plot(ax=ax)

In [ ]:
# access the single geometry object
seine = s_seine_utm.geometry.item()
seine

In [ ]:
seine.buffer(150)

In [ ]:
mask = districts.intersects(seine)
districts_seine = districts[mask]

In [ ]:
fig, ax = plt.subplots(figsize=(20, 10))
districts.plot(ax=ax, color='grey', alpha=0.4, edgecolor='k')
districts_seine.plot(ax=ax, color='blue', alpha=0.4, edgecolor='k')
s_seine_utm.plot(ax=ax)

------

<div class="alert alert-success">

**EXERCISE 12: Exploring a Land Use dataset**

For the following exercises, we first introduce a new dataset: a dataset about the land use of Paris (a simplified version based on the open European [Urban Atlas](https://land.copernicus.eu/local/urban-atlas)). The land use indicates for what kind of activity a certain area is used, such as residential area or for recreation. It is a polygon dataset, with a label representing the land use class for different areas in Paris.

In this exercise, we will read the data, explore it visually, and calculate the total area of the different classes of land use in the area of Paris.

* Read in the `'data/raw/paris_land_use'` shape file and assign the result to a variable `land_use`.
* Make a plot of `land_use`, using the `'class'` column to color the polygons. Add a legend with `legend=True`, and make the figure size a bit larger.
* Add a new column `'area'` to the dataframe with the area of each polygon.
* Calculate the total area in km² for each `'class'` using the `groupby()` method, and print the result.

<details><summary>Hints</summary>

* Reading a file can be done with the `geopandas.read_file()` function.
* To use a column to color the geometries, use the `column` keyword to indicate the column name.
* The area of each geometry can be accessed with the `area` attribute of the `geometry` of the GeoDataFrame.
* The `groupby()` method takes the column name on which you want to group as the first argument.

</details>

</div>

In [ ]:
land_use = gpd.read_file('data/raw/paris_land_use')
land_use.head()

In [ ]:
land_use.plot(column='class', figsize=(15, 10))

In [ ]:
land_use['area'] = land_use.geometry.area
land_use

In [ ]:
land_use.groupby('class')['area'].sum() /1000**2

<div class="alert alert-success">

**EXERCISE 13: Intersection of two polygons**

For this exercise, we are going to use 2 individual polygons: the district of Muette extracted from the `districts` dataset, and the green urban area of Boulogne, a large public park in the west of Paris, extracted from the `land_use` dataset. The two polygons have already been assigned to the `muette` and `park_boulogne` variables.

We first visualize the two polygons. You will see that they overlap, but the park is not fully located in the district of Muette. Let's determine the overlapping part:

* Plot the two polygons in a single map to examine visually the degree of overlap
* Calculate the intersection of the `park_boulogne` and `muette` polygons.
* Plot the intersection.
* Print the proportion of the area of the district that is occupied by the park.

<details><summary>Hints</summary>

* To plot single Shapely objects, you can put those in a `GeoSeries([..])` to use the GeoPandas `plot()` method.
* The intersection of to scalar polygons can be calculated with the `intersection()` method of one of the polygons, and passing the other polygon as the argument to that method.

</details>

</div>

In [ ]:
land_use = gpd.read_file("data/raw/paris_land_use")
districts = gpd.read_file("data/raw/paris_districts.geojson").to_crs(land_use.crs)

In [ ]:
# extract polygons
land_use['area'] = land_use.geometry.area
park_boulogne = land_use[land_use['class'] == "Green urban areas"].sort_values('area').geometry.iloc[-1]
muette = districts[districts.district_name == 'Muette'].geometry.item()

In [ ]:
# Plot the two polygons
gpd.GeoSeries([park_boulogne, muette]).plot(alpha=0.5, color=['green', 'blue'])

In [ ]:
intersection = park_boulogne.intersection(muette)

In [ ]:
gpd.GeoSeries([intersection]).plot()

In [ ]:
intersection.area / muette.area

<div class="alert alert-success">

**EXERCISE 14: Intersecting a GeoDataFrame with a Polygon**

Combining the land use dataset and the districts dataset, we can now investigate what the land use is in a certain district.

For that, we first need to determine the intersection of the land use dataset with a given district. Let's take again the *Muette* district as example case.

* Calculate the intersection of the `land_use` polygons with the single `muette` polygon. Call the result `land_use_muette`.
* Remove the empty geometries from `land_use_muette`.
* Make a quick plot of this intersection, and pass `edgecolor='black'` to more clearly see the boundaries of the different polygons.
* Print the first five rows of `land_use_muette`.

<details><summary>Hints</summary>

* The intersection of each geometry of a GeoSeries with another single geometry can be performed with the `intersection()` method of a GeoSeries.
* The `intersection()` method takes as argument the geometry for which to calculate the intersection.
* We can check which geometries are empty with the `is_empty` attribute of a GeoSeries.
    
</details>

</div>

In [ ]:
land_use = gpd.read_file("data/raw/paris_land_use")
districts = gpd.read_file("data/raw/paris_districts.geojson").to_crs(land_use.crs)
muette = districts[districts.district_name == 'Muette'].geometry.item()

In [ ]:
land_use_muette = land_use.intersection(muette)
land_use_muette

In [ ]:
land_use_muette = land_use_muette[~land_use_muette.is_empty]
land_use_muette

In [ ]:
land_use_muette.plot(edgecolor='black')

You can see in the plot that we now only have a subset of the full land use dataset. The original `land_use_muette` (before removing the empty geometries) still has the same number of rows as the original `land_use`, though. But many of the rows, as you could see by printing the first rows, consist now of empty polygons when it did not intersect with the Muette district.

The `intersection()` method also returned only geometries. If we want to combine those intersections with the attributes of the original land use, we can take a copy of this and replace the geometries with the intersections (you can uncomment and run to see the code):

In [ ]:
land_use_muette = land_use.copy()
land_use_muette['geometry'] = land_use.geometry.intersection(muette)
land_use_muette = land_use_muette[~land_use_muette.is_empty]
land_use_muette.head()

In [ ]:
land_use_muette.plot(column="class") #edgecolor="black")

In [ ]:
land_use_muette.dissolve(by='class')

<div class="alert alert-success">

**EXERCISE 15: The land use of the Muette district**

Based on the `land_use_muette` dataframe with the land use for the Muette districts as calculated above, we can now determine the total area of the different land use classes in the Muette district.

* Calculate the total area per land use class.
* Calculate the fraction (in percentage) for the different land use classes.

<details><summary>Hints</summary>

* The intersection of each geometry of a GeoSeries with another single geometry can be performed with the `intersection()` method of a GeoSeries.
* The `intersection()` method takes as argument the geometry for which to calculate the intersection.
* We can check which geometries are empty with the `is_empty` attribute of a GeoSeries.
    
</details>

</div>

In [ ]:
land_use_muette

In [ ]:
land_use_muette['area'] = land_use_muette.geometry.area
class_area = land_use_muette.groupby('class')['area'].sum()
fracs = land_use_muette.groupby('class')['area'].sum() / land_use_muette.geometry.area.sum() * 100


The above was only for a single district. If we want to do this more easily for all districts, we can do this with the overlay operation.

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## The overlay operation

In a spatial join operation, we are not changing the geometries itself. We are not joining geometries, but joining attributes based on a spatial relationship between the geometries. This also means that the geometries need to at least overlap partially.

If you want to create new geometries based on joining (combining) geometries of different dataframes into one new dataframe (eg by taking the intersection of the geometries), you want an **overlay** operation.

### How does it differ compared to the intersection method?

With the `intersection()` method introduced in the previous section, we could for example determine the intersection of a set of countries with another polygon, a circle in the example below:

<img width="70%" src="https://raw.githubusercontent.com/jorisvandenbossche/DS-python-geospatial/014bdb44211de665abf8960afcbdde429207f0cc/img/geopandas/chapter3-overlay-countries-circle-intersection-new.png"/>

However, this method (`countries.intersection(circle)`) also has some limitations.

* Mostly useful when intersecting a GeoSeries with a single polygon.
* Does not preserve attribute information of the intersecting polygons.

For cases where we require a bit more complexity, it is preferable to use the "overlay" operation, instead of the intersection method.

Consider the following simplified example. On the left we see again the 3 countries. On the right we have the plot of a GeoDataFrame with some simplified geologic regions for the same area:

<table width="80%"><tr>
<td> <img src="https://raw.githubusercontent.com/jorisvandenbossche/DS-python-geospatial/014bdb44211de665abf8960afcbdde429207f0cc/img/geopandas/chapter3-overlay-countries.png"/> </td>
<td> <img src="https://raw.githubusercontent.com/jorisvandenbossche/DS-python-geospatial/014bdb44211de665abf8960afcbdde429207f0cc/img/geopandas/chapter3-overlay-regions.png"/> </td>
</tr></table>

By simply plotting them on top of each other, as shown below, you can see that the polygons of both layers intersect.

But now, by "overlaying" the two layers, we can create a third layer that contains the result of intersecting both layers: all the intersections of each country with each geologic region. It keeps only those areas that were included in both layers.

<table width="80%"><tr>
<td> <img src="https://raw.githubusercontent.com/jorisvandenbossche/DS-python-geospatial/014bdb44211de665abf8960afcbdde429207f0cc/img/geopandas/chapter3-overlay-both.png"/> </td>
<td> <img src="https://raw.githubusercontent.com/jorisvandenbossche/DS-python-geospatial/014bdb44211de665abf8960afcbdde429207f0cc/img/geopandas/chapter3-overlay-overlayed.png"/> </td>
</tr></table>

This operation is called an intersection overlay, and in GeoPandas we can perform this operation with the `geopandas.overlay()` function.

Another code example:

In [ ]:
africa = countries[countries['continent'] == 'Africa']

In [ ]:
africa.plot()

In [ ]:
cities['geometry'] = cities.buffer(2)

In [ ]:
intersection = geopandas.overlay(africa, cities, how='intersection')
intersection.plot()

In [ ]:
intersection.head()

With the overlay method, we pass the full GeoDataFrame with all regions to intersect the countries with. The result contains all non-empty intersections of all combinations of countries and city regions.

Note that the result of the overlay function also keeps the attribute information of both the countries as the city regions. That can be very useful for further analysis.

In [ ]:
geopandas.overlay(africa, cities, how='intersection').plot()  # how="difference"/"union"/"symmetric_difference"

<div class="alert alert-info" style="font-size:120%">
<b>REMEMBER</b> <br>

* **Spatial join**: transfer attributes from one dataframe to another based on the spatial relationship
* **Spatial overlay**: construct new geometries based on spatial operation between both dataframes (and combining attributes of both dataframes)

</div>

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Let's practice!

<!-- BEGIN QUESTION -->

<div class="alert alert-success">

**EXERCISE 16: Overlaying spatial datasets I**

We will now combine both datasets in an overlay operation. Create a new `GeoDataFrame` consisting of the intersection of the land use polygons which each of the districts, but make sure to bring the attribute data from both source layers.

* Create a new GeoDataFrame from the intersections of `land_use` and `districts`. Assign the result to a variable `combined`.
* Print the first rows the resulting GeoDataFrame (`combined`).

<details><summary>Hints</summary>

* The intersection of two GeoDataFrames can be calculated with the `geopandas.overlay()` function.
* The `overlay()` functions takes first the two GeoDataFrames to combine, and a third `how` keyword indicating how to combine the two layers.
* For making an overlay based on the intersection, you can pass `how='intersection'`.
* The default `overlay()` call will generate a warning because some of the intersections result in a point or linestring, and thus not in a new polygon. For this exercise, we are only interested in the resulting polygons, and therefore we set `keep_geom_type=True` to suppress the warning and say to `overlay()` we only need the resulting geometries of the same type as the original geometries (thus, only polygons).

</details>

</div>

In [ ]:
land_use = geopandas.read_file("data/raw/paris_land_use")
districts = geopandas.read_file("data/raw/paris_districts.geojson").to_crs(land_use.crs)

<div class="alert alert-success">

**EXERCISE 17: Overlaying spatial datasets II**

Now that we created the overlay of the land use and districts datasets, we can more easily inspect the land use for the different districts. Let's get back to the example district of Muette, and inspect the land use of that district.

* Add a new column `'area'` with the area of each polygon to the `combined` GeoDataFrame.
* Create a subset called `land_use_muette` where the `'district_name'` is equal to "Muette".
* Make a plot of `land_use_muette`, using the `'class'` column to color the polygons.
* Calculate the total area for each `'class'` of `land_use_muette` using the `groupby()` method, and print the result.

<details><summary>Hints</summary>

* The area of each geometry can be accessed with the `area` attribute of the `geometry` of the GeoDataFrame.
* To use a column to color the geometries, pass its name to the `column` keyword.
* The `groupby()` method takes the column name on which you want to group as the first argument.
* The total area for each class can be calculated by taking the `sum()` of the area.

</details>

</div>

<div class="alert alert-success">

**EXERCISE 18: Overlaying spatial datasets III**

Thanks to the result of the overlay operation, we can now more easily perform a similar analysis for *all* districts. Let's investigate the fraction of green urban area in each of the districts.

* Based on the `combined` dataset, calculate the total area per district using `groupby()`.
* Select the subset of "Green urban areas" from `combined` and call this `urban_green`.
* Now calculate the total area per district for this `urban_green` subset, and call this `urban_green_area`.
* Determine the fraction of urban green area in each district.
* Check the districts with the largest and smallest fraction of urban green (you can sort the result, or use `nlargest()` / `nsmallest()`)

</div>

<!-- END QUESTION -->



## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

Upload your submission zip to **LAB07 - GeoPandas II & III** on [Gradescope](https://www.gradescope.com/).

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)